In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Pertemuan7-StudiKasusTerintegrasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [18]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["DImas", "Delyn", "Marsha"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggahtop ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [19]:
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

print("Data berhasil di extrak!")

Data berhasil di extrak!


In [20]:
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                          .join(df_menu, on="menu_id", how="inner") 
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

df_gabungan.show(5)
                                     

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|        Marsha|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|        Marsha|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|        Marsha|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|        Marsha|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|         Delyn|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [25]:
ringkasan_menu = df_gabungan.groupBy("nama_cabang", "nama_menu").agg (
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())
top2_menu = ringkasan_menu.withColumn("peringkat", row_number().over(window_cabang)) \
                          .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show()

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



In [27]:
df_gabungan.createOrReplaceTempView("data_kopi")

kontribusi_barista = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM data_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')

kontribusi_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|        Marsha|      177680000|
|         Delyn|       84645000|
+--------------+---------------+



In [31]:
path_simpan = "hdfs://localhost:9000/user/dimas/tugas7/processed/laporan"
df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(path_simpan)

df_verifikasi = spark.read.parquet(path_simpan)
print("verifikasi load - total baris data tersimpan:", df_verifikasi.count())

verifikasi load - total baris data tersimpan: 4000


In [34]:
ringkasan_eksekutif = df_gabungan.groupBy("nama_cabang").agg(
    spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_triwulan").desc())

ringkasan_eksekutif.show()

+--------------------+------------------------+----------------+-------------------------+
|         nama_cabang|total_penjualan_triwulan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+------------------------+----------------+-------------------------+
|Kopi Nusantara Se...|                90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|                87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|                84645000|            1305|        64862.06896551724|
+--------------------+------------------------+----------------+-------------------------+



Berdasarkan hasil analisis data end-to-end menggunakan ekosistem Big Data, ditemukan bahwa kinerja penjualan antarcabang memiliki variasi yang cukup signifikan. (Misalnya: Kopi Nusantara Yogyakarta menunjukkan performa terbaik dengan penyumbang omzet total terbesar). Sebaliknya, (Kopi Nusantara Semarang) saat ini tercatat sebagai cabang dengan tingkat kinerja penjualan (total pendapatan dan jumlah transaksi) yang paling lemah atau berada di posisi terbawah dibandingkan cabang lainnya.

Untuk mendongkrak penjualan di cabang dengan kinerja terlemah tersebut, saya merekomendasikan strategi promosi agresif yang difokuskan pada produk Top-2 best-selling mereka. Berdasarkan analisis window function, menu andalan yang paling diminati di cabang tersebut adalah (sebutkan Menu 1 dari output tabel C) dan (sebutkan Menu 2 dari output tabel C). Manajemen sebaiknya membuat program "bundling promo" atau diskon khusus "Buy 1 Get 1" pada kedua menu primadona ini di jam-jam sibuk. Alasan utamanya adalah karena kedua menu ini sudah terbukti memiliki minat dasar (demand) yang kuat dari konsumen lokal. Mempromosikan produk yang sudah jelas diminati jauh lebih meminimalisir risiko bakar uang ketimbang mempromosikan produk yang kurang laku, sehingga dapat menarik traffic pelanggan baru dan berpotensi meningkatkan nilai rata-rata transaksi harian secara lebih cepat dan efektif.